# ComfyUI Wan2.2 Animate 14B — Colab 一键运行

运行本项目的 `api.json` 工作流（Wan2.2 Animate 14B GGUF 视频换人/动作迁移）。

**使用前**: 菜单 `运行时 → 更改运行时类型 → T4 GPU`。

**步骤**: 从上往下依次运行 6 个单元格。第 5 格会弹出文件选择框，需要上传：
- `api.json`（工作流文件）
- 参考图 `78515578_109_5e22.jpg`（**文件名必须与工作流里一致**）
- 输入视频 `.mp4`（**文件名必须与工作流里一致**）

> 注意: Colab 免费版单次会话最长约 12 小时（常会提前断开），断开即清空。
> 模型共约 15GB，首次运行下载需 10~30 分钟；T4 上采样较慢，请耐心。

In [ ]:
# ===== 1/6 检查 GPU =====
import torch
assert torch.cuda.is_available(), "请先在 运行时 → 更改运行时类型 里选择 T4 GPU!"
!nvidia-smi


In [ ]:
# ===== 2/6 安装 ComfyUI 本体 =====
import os, subprocess, sys

COMFY = "/content/ComfyUI"
if not os.path.isdir(COMFY):
    subprocess.run(["git", "clone", "--depth", "1",
                    "https://github.com/comfyanonymous/ComfyUI.git", COMFY], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r",
                f"{COMFY}/requirements.txt"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "websocket-client", "huggingface_hub[hf_transfer]"], check=True)
print("ComfyUI 安装完成")


In [ ]:
# ===== 3/6 安装工作流用到的自定义节点 =====
import glob, os, subprocess, sys

CUSTOM_NODES = [
    "https://github.com/ltdrdata/ComfyUI-Manager",               # 管理器(可选)
    "https://github.com/rgthree/rgthree-comfy",                  # Any Switch / Power Lora Loader
    "https://github.com/ltdrdata/ComfyUI-Impact-Pack",           # BboxDetectorSEGS / SegsToCombinedMask ...
    "https://github.com/ltdrdata/ComfyUI-Impact-Subpack",        # UltralyticsDetectorProvider
    "https://github.com/Fannovel16/comfyui_controlnet_aux",      # DWPreprocessor
    "https://github.com/Kosinkadink/ComfyUI-VideoHelperSuite",   # VHS_LoadVideo / VHS_VideoCombine
    "https://github.com/kijai/ComfyUI-KJNodes",                  # ImageResizeKJv2 / PathchSageAttentionKJ / BlockifyMask / DrawMaskOnImage
    "https://github.com/city96/ComfyUI-GGUF",                    # UnetLoaderGGUF
    "https://github.com/pythongosssss/ComfyUI-Custom-Scripts",   # MathExpression|pysssss
    "https://github.com/yolain/ComfyUI-Easy-Use",                # easy indexAnything
    "https://github.com/chflame163/ComfyUI_LayerStyle",          # LayerUtility: PurgeVRAM
    "https://github.com/chflame163/ComfyUI_LayerStyle_Advance",  # LayerUtility: PurgeVRAM V2
    "https://github.com/1038lab/ComfyUI-RMBG",                   # ClothesSegment
    "https://github.com/jamesWalker55/comfyui-various",          # JWImageResizeByShorterSide
    "https://github.com/melMass/comfy_mtb",                      # PixelPerfectResolution
]

nodes_dir = f"{COMFY}/custom_nodes"
os.makedirs(nodes_dir, exist_ok=True)
for repo in CUSTOM_NODES:
    dest = os.path.join(nodes_dir, os.path.basename(repo))
    if not os.path.isdir(dest):
        subprocess.run(["git", "clone", "--depth", "1", repo, dest])
for req in glob.glob(f"{nodes_dir}/*/requirements.txt"):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", req])
print("自定义节点安装完成")


In [ ]:
# ===== 4/6 下载模型 (约 15GB, 需 10~30 分钟) =====
import os, shutil
from huggingface_hub import hf_hub_download

os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
MODELS = [
    ("QuantStack/Wan2.2-Animate-14B-GGUF", "Wan2.2-Animate-14B-Q2_K.gguf",
     f"{COMFY}/models/diffusion_models"),
    ("Comfy-Org/Wan_2.1_ComfyUI_repackaged", "split_files/text_encoders/umt5_xxl_fp16.safetensors",
     f"{COMFY}/models/text_encoders"),
    ("Comfy-Org/Wan_2.1_ComfyUI_repackaged", "split_files/vae/wan_2.1_vae.safetensors",
     f"{COMFY}/models/vae"),
    ("Comfy-Org/Wan_2.1_ComfyUI_repackaged", "split_files/clip_vision/clip_vision_h.safetensors",
     f"{COMFY}/models/clip_vision"),
    ("Bingsu/adetailer", "face_yolov8m.pt",
     f"{COMFY}/models/ultralytics/bbox"),
    ("hr16/yolox-onnx", "yolox_l.torchscript.pt",
     f"{COMFY}/custom_nodes/comfyui_controlnet_aux/ckpts/hr16/yolox-onnx"),
    ("hr16/DWPose-TorchScript-BatchSize5", "dw-ll_ucoco_384_bs5.torchscript.pt",
     f"{COMFY}/custom_nodes/comfyui_controlnet_aux/ckpts/hr16/DWPose-TorchScript-BatchSize5"),
]
for repo, filename, dest_dir in MODELS:
    os.makedirs(dest_dir, exist_ok=True)
    target = os.path.join(dest_dir, os.path.basename(filename))
    if os.path.exists(target):
        print("已存在, 跳过:", filename, flush=True)
        continue
    print(f"下载 {repo}/{filename}", flush=True)
    shutil.copy(hf_hub_download(repo, filename), target)
print("全部模型下载完成")


In [ ]:
# ===== 5/6 上传 api.json 和输入素材 =====
# 弹出文件选择框, 请一次选中: api.json + 参考图 + 输入视频
# (文件名必须与工作流里写的一致; 视频较大时建议改从 Google Drive 拷贝)
from google.colab import files

os.makedirs(f"{COMFY}/input", exist_ok=True)
uploaded = files.upload()
for name, data in uploaded.items():
    dest = "/content/api.json" if name == "api.json" else f"{COMFY}/input/{name}"
    with open(dest, "wb") as f:
        f.write(data)
    print("保存:", dest)
print("input 目录:", os.listdir(f"{COMFY}/input"))


In [ ]:
# ===== 6/6 启动 ComfyUI 并运行工作流 =====
# T4 16GB 显存较紧, 如 OOM 在启动参数里加 "--lowvram"
import json, subprocess, sys, time, urllib.request

PORT = 8188
log = open("/content/comfyui.log", "w")
server = subprocess.Popen(
    [sys.executable, "main.py", "--port", str(PORT), "--disable-auto-launch"],
    cwd=COMFY, stdout=log, stderr=subprocess.STDOUT)

for _ in range(120):
    try:
        urllib.request.urlopen(f"http://127.0.0.1:{PORT}/system_stats", timeout=2)
        break
    except Exception:
        time.sleep(5)
else:
    print(open("/content/comfyui.log").read()[-3000:])
    raise SystemExit("ComfyUI 启动超时")
print("ComfyUI 已启动", flush=True)

wf = json.load(open("/content/api.json"))

def post(path, payload):
    req = urllib.request.Request(f"http://127.0.0.1:{PORT}{path}",
                                 data=json.dumps(payload).encode(),
                                 headers={"Content-Type": "application/json"})
    return json.load(urllib.request.urlopen(req))

prompt_id = post("/prompt", {"prompt": wf})["prompt_id"]
print("prompt_id:", prompt_id, flush=True)

outputs = []
while True:
    time.sleep(10)
    history = json.load(urllib.request.urlopen(
        f"http://127.0.0.1:{PORT}/history/{prompt_id}"))
    if prompt_id not in history:
        print("排队/执行中...", flush=True)
        continue
    entry = history[prompt_id]
    status = entry.get("status", {})
    if status.get("status_str") == "error":
        print("执行出错:", json.dumps(status.get("messages", []), ensure_ascii=False))
        break
    if status.get("completed"):
        for node_out in entry.get("outputs", {}).values():
            for key in ("images", "gifs", "videos"):
                for item in node_out.get(key, []):
                    url = (f"http://127.0.0.1:{PORT}/view?filename={item['filename']}"
                           f"&subfolder={item.get('subfolder','')}&type={item.get('type','output')}")
                    dest = f"/content/{item['filename']}"
                    with open(dest, "wb") as f:
                        f.write(urllib.request.urlopen(url).read())
                    outputs.append(dest)
                    print("已保存:", dest, flush=True)
        break

server.terminate()
log.close()

# 自动下载结果到本地浏览器
for f in outputs:
    files.download(f)
print("完成!")
